# Authentic exercise: accumulate telescope samples into sky pixels

**Optional deep dive, about 20 minutes.** Astronomy instruments record a pixel index and measurement at each timestamp. We will sum measurements by pixel, first with pandas and then with a compiled loop.

In [ ]:
import os
import time
import numpy as np
import pandas as pd
from numba import njit

test_mode = os.environ.get("PYHPC_TEST_MODE") == "1"
n_pixels = 50_000
n_samples = 200_000 if test_mode else 5_000_000
rng = np.random.default_rng(2026)
pixels = rng.integers(0, n_pixels, size=n_samples, dtype=np.int64)
timeline = rng.normal(size=n_samples)
print(f"Samples: {n_samples:,}")

## 1. Establish a library baseline

In [ ]:
started = time.perf_counter()
pandas_result = (
    pd.Series(timeline, index=pixels)
    .groupby(level=0)
    .sum()
    .reindex(range(n_pixels), fill_value=0.0)
    .to_numpy()
)
pandas_time = time.perf_counter() - started
print(f"pandas: {pandas_time:.3f} s")

## 2. Write the loop

For every sample, add its value to the corresponding output pixel.

```python
def accumulate_samples(pixel_index, values, output):
    for sample in range(pixel_index.size):
        # Add values[sample] to the selected output pixel.
```

Write and test your loop in the next cell. Use a small slice first so a mistake is quick to diagnose.

In [ ]:
# Write and test your loop here before running the solution below.

### Solution and small-input correctness check

Run the next cell after attempting the loop. The small test compares against `np.bincount`, which provides an independent reference result.

In [ ]:
def accumulate_samples_python(pixel_index, values, output):
    for sample in range(pixel_index.size):
        output[pixel_index[sample]] += values[sample]


small_count = min(20_000, n_samples)
small_output = np.zeros(n_pixels)
accumulate_samples_python(pixels[:small_count], timeline[:small_count], small_output)
np.testing.assert_allclose(
    small_output,
    np.bincount(pixels[:small_count], weights=timeline[:small_count], minlength=n_pixels),
    rtol=1e-12,
)

Running the pure-Python loop over all samples would be a poor use of classroom time. Compile the same function, warm it up, and then measure the full input.

In [ ]:
accumulate_samples_numba = njit(accumulate_samples_python)
warmup_output = np.zeros(n_pixels)
accumulate_samples_numba(pixels[:10], timeline[:10], warmup_output)

numba_result = np.zeros(n_pixels)
started = time.perf_counter()
accumulate_samples_numba(pixels, timeline, numba_result)
numba_time = time.perf_counter() - started

np.testing.assert_allclose(numba_result, pandas_result, rtol=1e-12, atol=1e-12)
print(f"Numba:  {numba_time:.3f} s")
print(f"pandas / Numba: {pandas_time / numba_time:.1f}x")

## Takeaway

Irregular indexed accumulation is a useful Numba target. Expressing it as a chain of vectorized NumPy operations would create extra work or temporary arrays. Numba preserves the direct loop and removes Python interpreter overhead.

Real library examples: [PySM power law](https://github.com/galsci/pysm/blob/2b69973bc8d6bd7a9d5c861477d015461fde185a/pysm3/models/power_law.py#L101-L132) and [PySM dust](https://github.com/galsci/pysm/blob/2b69973bc8d6bd7a9d5c861477d015461fde185a/pysm3/models/dust.py#L109-L157).